In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Add, Concatenate
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
import tensorflow as tf

#Viene definita la struttura della ResNet, dove:
    # Dense(factors) -> crea un layer fully connected con 256 neuroni;
    # ReLu -> per la non linearità;
    # BatchNormalization -> stabilizza l'addestramento;
    # Dropout -> riduce l'overfitting spegnendo casualmente una porzione di neuroni in fase di training
    
def create_resnet(input_dim, num_classes, dropout_pcg=0.25, factors=256):
    inp = Input(shape=(input_dim,))
    #Primo blocco "base"
    out = Dense(factors, activation="relu")(inp)
    out = BatchNormalization()(out)
    out = Dropout(dropout_pcg)(out)

    #Blocco "residuale"
    res = Dense(factors, activation="relu")(out)
    res = BatchNormalization()(res)
    res = Dropout(dropout_pcg)(res)

    #Skip connection (somma elemento per elemento out+res)
    out = Add()([out, res])
    
    #Concatenazione con l'input originale
    out = Concatenate()([out, inp])

    # nell'Output si usa Softmax per ottenere una distribuzione di probabilità sulle classi
    out = Dense(num_classes, activation="softmax")(out)

    m = Model(inp, out)
    m.compile(
        loss="categorical_crossentropy",
        optimizer=tf.keras.optimizers.RMSprop(1e-3),
        metrics=["accuracy"]
    )
    return m


# ==========================
# Inserimento Dataset singolo soggetto
# ==========================
FILE_PATH = "PSD_dataset_Capria_Francesco_26.csv" 
OUT_DIR = "single_subject_resnet"
os.makedirs(OUT_DIR, exist_ok=True)

# ==========================
# Carimanto Dataset
# ==========================
df = pd.read_csv(FILE_PATH)

# Feature PSD (tutte le colonne tranne Event ID)
X = df.drop("Event Id", axis=1).astype(float).values
# Etichette (Event Id)
y = df["Event Id"].values

input_dim = X.shape[1]  #N° Feature

# classi del singolo soggetto (mapping coerente train/test)
# le label originali vengono convertite in indici (perchè "to_categorical" necessita di indici consecutivi)
classes = np.array(sorted(np.unique(y)))
num_classes = len(classes)
to_index = {c: i for i, c in enumerate(classes)}


# ==========================
# SPLIT train/test
# ==========================
# Train -> 80%
# Test -> 20%
# Shuffle mischia il dataset, ma Stratify mantiene la stessa proporzione di classi in train e test
Xtr, Xte, ytr, yte = train_test_split(
    X, y,
    train_size=0.8,
    shuffle=True,
    stratify=y
)

# ==========================
# SCALER
# ==========================
# fit SOLO sul train per evitare Data Leakage
scaler = MinMaxScaler().fit(Xtr)
Xtr_s = scaler.transform(Xtr)
Xte_s = scaler.transform(Xte)

# ==========================
# OVERSAMPLING sul train
# ==========================
# SOLO sul train per evitare Data Leakage
df_tr = pd.DataFrame(Xtr_s)
df_tr["Event Id"] = ytr

max_n = df_tr["Event Id"].value_counts().max()
dfs = []
for cls, sub in df_tr.groupby("Event Id"):
    if len(sub) < max_n:
        sub = resample(sub, replace=True, n_samples=max_n)
    dfs.append(sub)

df_bal = pd.concat(dfs).sample(frac=1).reset_index(drop=True)
Xtr_bal = df_bal.drop("Event Id", axis=1).values
ytr_bal = df_bal["Event Id"].values

# ==========================
# ONE-HOT (solo classi del soggetto)
# ==========================
ytr_cat = to_categorical([to_index[c] for c in ytr_bal], num_classes=num_classes)
yte_idx = np.array([to_index[c] for c in yte])

# ==========================
# TRAIN
# ==========================
# ModelCheckpoint salva i pesi migliori (quelli con val_loss minima);
# ReduceLROnPlateau abbassa il learning rate se la validazione non migliora;
model = create_resnet(input_dim, num_classes)

ckpt_path = os.path.join(OUT_DIR, "best_resnet.weights.h5")
ckpt = ModelCheckpoint(
    ckpt_path,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    mode="min",
    verbose=0
)
rlr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=0
)

# sul train bilanciato si usa il 20% come validation set
model.fit(
    Xtr_bal, ytr_cat,
    validation_split=0.2,
    epochs=150,
    batch_size=64,
    callbacks=[ckpt, rlr],
    verbose=0     # verbose = 1 per stampare le epoche
)

# ==========================
# TEST
# ==========================
# Si prendono i pesi migliori, si effettua la predizione 
#sul dataset scalato e poi argmax fornisce la classse predetta
model.load_weights(ckpt_path)
ypred = model.predict(Xte_s, verbose=0)
ypred_lab = np.argmax(ypred, axis=1)

acc = accuracy_score(yte_idx, ypred_lab)
print(f"\nTest Accuracy: {acc:.4f}")
print(classification_report(
    yte_idx, ypred_lab,
    target_names=[str(c) for c in classes]
))


Epoch 1/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.4179 - loss: 1.6341 - val_accuracy: 0.3809 - val_loss: 1.4698 - learning_rate: 0.0010
Epoch 2/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7503 - loss: 0.6680 - val_accuracy: 0.2852 - val_loss: 1.4277 - learning_rate: 0.0010
Epoch 3/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8556 - loss: 0.4050 - val_accuracy: 0.3184 - val_loss: 1.3174 - learning_rate: 0.0010
Epoch 4/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9121 - loss: 0.3106 - val_accuracy: 0.2910 - val_loss: 1.2727 - learning_rate: 0.0010
Epoch 5/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9388 - loss: 0.2104 - val_accuracy: 0.5391 - val_loss: 1.0541 - learning_rate: 0.0010
Epoch 6/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9621 - loss: 0.1443 - val_accuracy: 0.7559 - val_loss: 0.8694 - learning_rate: 0.0010
Epoch 7/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9659 - loss: 0.1293 - val_ac